# Example 1 — Synthetic GNSS Time Series

This notebook reproduces the full Example 1 workflow from the Hector manual:

1. **Remove outliers** — flag spike outliers in the raw time series
2. **Estimate trend and noise** — fit a linear trend + annual/semi-annual
   signals using GGM+White noise via RMLE
3. **Plot the fitted series** — visualise the data with the estimated model
4. **Power spectral density** — compare the residual PSD with the fitted
   noise model

**Before running:** copy the examples to a working directory with
`hector-examples` and open this notebook from inside `ex1/`.

In [ ]:
import os
import json
import subprocess
import numpy as np
import matplotlib.pyplot as plt

from hector.control import Control, SingletonMeta
from hector.observations import Observations
from hector.designmatrix import DesignMatrix
from hector.covariance import Covariance
from hector.mle import MLE

%matplotlib inline

# This notebook must be run with ex1/ as the working directory.
# If Jupyter was launched from elsewhere, adjust the path below.
# os.chdir('/path/to/hector-examples/ex1')

## Step 1 — Remove Outliers

The raw data in `obs_files/TEST.mom` contains spike outliers.
`removeoutliers` flags them using the IQ-factor criterion and writes
the cleaned series to `pre_files/TEST.mom`.

In [ ]:
result = subprocess.run(['removeoutliers'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
with open('removeoutliers.json') as f:
    ro = json.load(f)

print(f"Observations : {ro['N']}")
print(f"Outliers removed : {len(ro['outliers'])}")
print("Outlier epochs (MJD):", ro['outliers'])

## Step 2 — Estimate Trend and Noise Parameters

`estimatetrend` fits a linear trajectory with annual and semi-annual
periodic signals and four offsets at the epochs given in the
`obs_files/TEST.mom` header.  Noise is modelled as GGM + White.
RMLE is used to remove bias in the noise parameter estimates.

In [ ]:
SingletonMeta.clear_all()

control = Control('estimatetrend.ctl')
obs     = Observations()
des     = DesignMatrix()
cov     = Covariance()
mle     = MLE()

theta, C_theta, noise_params, sigma_eta = mle.estimate_parameters()
error = np.sqrt(np.diagonal(C_theta))

output = {}
obs.show_results(output)
mle.show_results(output)
cov.show_results(output, noise_params, sigma_eta)
des.show_results(output, theta, error)

In [ ]:
print(f"Trend        : {output['trend']:.3f} ± {output['trend_sigma']:.3f} {output['PhysicalUnit']}/yr")
print(f"Annual amp   : {output['amp_365.250']:.3f} ± {output['amp_365.250_sigma']:.3f} {output['PhysicalUnit']}")
print()
print(f"AIC  : {output['AIC']:.2f}")
print(f"BIC  : {output['BIC']:.2f}")
print(f"KIC  : {output['KIC']:.2f}")
print()
for model, params in output['NoiseModel'].items():
    sigma = params.get('sigma', float('nan'))
    frac  = params.get('fraction', float('nan'))
    print(f"{model:8s}  sigma={sigma:.4f}  fraction={frac:.4f}")
print()
for epoch, size, sig in zip(output['jump_epochs'],
                             output['jump_sizes'],
                             output['jump_sigmas']):
    print(f"Offset  MJD {epoch:.1f}  size={size:+.2f} ± {sig:.2f} {output['PhysicalUnit']}")

## Step 3 — Plot Observed and Fitted Series

In [ ]:
des.add_mod(theta)   # writes fitted model into obs.data['mod']

mjd  = obs.data.index.to_numpy()
t    = (mjd - 51544) / 365.25 + 2000   # MJD → decimal year
x    = obs.data['obs'].to_numpy()
xhat = obs.data['mod'].to_numpy()
unit = output['PhysicalUnit']

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(t, x,    color='steelblue', lw=0.7, label='observed')
ax.plot(t, xhat, color='tomato',    lw=1.4, label='model')

# Mark offset epochs
for epoch in output['jump_epochs']:
    t_jump = (epoch - 51544) / 365.25 + 2000
    ax.axvline(t_jump, color='gray', lw=0.8, ls='--', alpha=0.6)

ax.set_xlabel('Year')
ax.set_ylabel(f'Position [{unit}]')
ax.set_title('Example 1 — observed vs. fitted')
ax.legend()
fig.tight_layout()
plt.savefig('data_figures/TEST.png', dpi=150)
plt.show()

## Step 4 — Residuals

In [ ]:
residuals = x - xhat

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(t, residuals, color='steelblue', lw=0.7)
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('Year')
ax.set_ylabel(f'Residual [{unit}]')
ax.set_title('Residuals')
fig.tight_layout()
plt.show()

print(f"RMS residual : {np.sqrt(np.nanmean(residuals**2)):.3f} {unit}")

## Step 5 — Power Spectral Density

`estimatespectrum` computes the Welch periodogram of the residuals.
`modelspectrum` evaluates the theoretical noise PSD from the estimated
noise parameters.  Both programs write a two-column text file
(frequency in Hz, power in unit²/Hz).

In [ ]:
subprocess.run(['estimatespectrum'], capture_output=True)
subprocess.run(['modelspectrum'],    capture_output=True)

In [ ]:
psd_data  = np.loadtxt('estimatespectrum.out')
psd_model = np.loadtxt('modelspectrum.out')

# Skip the zero-frequency bin
freq_data  = psd_data[1:, 0]
power_data = psd_data[1:, 1]
freq_model  = psd_model[:, 0]
power_model = psd_model[:, 1]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(freq_data,  power_data,  color='steelblue', lw=0.8, label='Welch periodogram')
ax.loglog(freq_model, power_model, color='tomato',    lw=1.6, label='GGM+White model')
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel(f'PSD [{unit}²/Hz]')
ax.set_title('Power spectral density of residuals')
ax.legend()
fig.tight_layout()
plt.savefig('psd_figures/TEST.png', dpi=150)
plt.show()